    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.
    
    Task 5 (LS3): Implement a program which, (a) given one of the feature models and (b) a value k,
    – creates (and saves) a label-label similarity matrix,
    – performs a user selected dimensionality reduction technique (SVD, NNMF, LDA, k-means) on this label-label
    similarity matrix,
    – stores the latent semantics in a properly named output file
    – lists label-weight pairs, ordered in decreasing order of weights

In [43]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))


In [44]:
from utils.database_utils import retrieve
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

print("Generating top-", K, " label latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 5  label latent semantics under  resnet_output  feature space using:  kmeans


In [45]:
from utils.database_utils import exists, retrieve, store
from feature_models.feature_matrix.label_label_similarity import LabelLabelSimilarity
from utils.dataset_utils import initialize_dataset

if exists(f'label_label_{FEATURE_SPACE}.pt'):

    label_feature_vectors = retrieve(f'label_label_{FEATURE_SPACE}.pt')

else:
    print('Label-label similiarity matrix for ', FEATURE_SPACE, ' does not exist, creating one.')

    label_similarity_generator = LabelLabelSimilarity(feature_vectors, initialize_dataset().categories)
    label_feature_vectors = label_similarity_generator.get_matrix()

    store(label_feature_vectors, f'label_label_{FEATURE_SPACE}.pt')

print("Label-label similarity matrix for faces as example:\n")
print(label_feature_vectors[0])
print("Shape: ", label_feature_vectors[0][1].shape)

Label-label similarity matrix for faces as example:

('Faces', array([1.00000000e+00, 9.60584402e-01, 6.55803531e-02, 0.00000000e+00,
       1.54499143e-01, 0.00000000e+00, 3.31209227e-02, 1.09157033e-01,
       6.65913522e-02, 9.91684571e-02, 0.00000000e+00, 2.38687266e-02,
       0.00000000e+00, 1.93770945e-01, 0.00000000e+00, 6.85443208e-02,
       1.32241055e-01, 5.58832437e-02, 3.02741174e-02, 0.00000000e+00,
       1.44533142e-01, 2.38082007e-01, 0.00000000e+00, 1.36598218e-02,
       2.75379769e-03, 1.77518219e-01, 8.92516747e-02, 8.64985138e-02,
       0.00000000e+00, 8.83039844e-04, 1.92280278e-01, 9.32242721e-02,
       2.80332714e-01, 0.00000000e+00, 1.55025363e-01, 2.40313053e-01,
       0.00000000e+00, 0.00000000e+00, 1.41856819e-01, 0.00000000e+00,
       0.00000000e+00, 3.00697256e-02, 1.07354060e-01, 3.03628832e-01,
       0.00000000e+00, 9.88438204e-02, 3.76361534e-02, 0.00000000e+00,
       1.71870902e-01, 5.14730737e-02, 0.00000000e+00, 0.00000000e+00,
       3.00716

In [46]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer


elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(label_feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(label_feature_vectors)

latent_semantics = reducer.reduce_features(label_feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[13.21005983 10.60808622 13.63693331 11.35669357 11.8781926 ]
 [13.01166898 10.91400083 13.52472754 11.36971437 12.13142933]
 [12.4678414  11.46623624  8.09350276 10.27546872 11.76621454]
 [ 9.04693941 12.50319486 12.28701435 11.52944382 13.56127935]
 [12.87312562  8.50322337 12.04278702 10.10884541 11.77246476]
 [ 7.98590717 12.13153102 11.77056528 10.43864054 12.64642874]
 [14.0381595   8.06935731 14.47374277 12.26154273 11.9126444 ]
 [15.29506846  9.07803951 14.29267811 10.34525652 10.0900305 ]
 [12.05261382  8.20514712 12.65906646 10.71182761  8.21409044]
 [13.76988299 10.47815549 10.93417031  7.86655483  9.75244775]
 [13.60483893 13.95216788  8.37594733  9.5626528  12.61017882]
 [11.83779418  9.05427399 12.14755877 11.06861371 13.15677824]
 [ 9.83645247 12.68067818 11.9303515  11.37764238  9.50968749]
 [15.79333162  9.55208894 13.32241986 10.01629883  8.0008006 ]
 [15.2439465  13.81064038  7.65590613 10.72003901 12.21705948]
 [12.79177849  8.81514864 11.9

In [47]:
# Store the latent semantics in a properly named file.
# We opt to store just the reducer, as we anyway can generate the latent space quickly
# by loading the feature space and passing it to the reducer, eg:
#
# unpicked_reducer = retrieve(f'LS3_color_svd_reducer.pt')
# feature_vectors = retrieve(f'color.pt')
#
# unpickled_reducer.reduce_features(feature_vectors)

store(reducer, f'LS3_{FEATURE_SPACE}_{K}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS3_resnet_output_5_kmeans_reducer.pt 



In [48]:
# List label-weight pairs, ordered in decreasing order of weights

# We are to showcase which labels contribute more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

labels = [feature_tuple[0] for feature_tuple in label_feature_vectors.values()]

label_weight_tuples = list(zip(labels, similarity_matrix))

print("Label - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for LABEL, weight in sorted(label_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(Label: ", LABEL, ", Weight: ", weight[i], end="),\t")

Label - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(Label:  octopus , Weight:  20.01717559366013),	(Label:  menorah , Weight:  17.069155374846254),	(Label:  garfield , Weight:  16.781800846633114),	(Label:  dragonfly , Weight:  16.63446234267021),	(Label:  snoopy , Weight:  16.458027828037405),	(Label:  yin_yang , Weight:  16.397682255508673),	(Label:  scissors , Weight:  16.11897874606846),	(Label:  butterfly , Weight:  15.936366919351816),	(Label:  brain , Weight:  15.793331624661203),	(Label:  dollar_bill , Weight:  15.365658700917374),	(Label:  gramophone , Weight:  15.321991278259308),	(Label:  ant , Weight:  15.295068455772112),	(Label:  brontosaurus , Weight:  15.243946501342844),	(Label:  wrench , Weight:  15.116123745638715),	(Label:  metronome , Weight:  15.102714023366811),	(Label:  stegosaurus , Weight:  14.807198482778535),	(Label:  lobster , Weight:  14.631742196838193),	(Label:  stapler , Weight:  14.595245541571721),